The predictions form teh from scratch model aren't great at all either, very similar to the previous models with pre-trained networks as base weights. The better predictions are from the pre-trained TN model as the base network. 

In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
import os
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import pearsonr

2025-03-12 10:40:32.359455: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-03-12 10:40:32.749816: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-03-12 10:40:32.749867: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-03-12 10:40:32.843571: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-03-12 10:40:32.971113: I tensorflow/core/platform/cpu_feature_guar

In [2]:
# load the models required
frozen_model = tf.keras.models.load_model('models/CNN_LSTM_from_scratch.keras')

2025-03-12 10:41:01.648479: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 31134 MB memory:  -> device: 0, name: Tesla V100S-PCIE-32GB, pci bus id: 0000:06:00.0, compute capability: 7.0


In [3]:
# load a test image to get the height and width information
file_path = 'All_data/block_0103/all_np_files/Block0103_2020_08_26.npy'

loaded_test_image = np.load(file_path)

print(loaded_test_image.shape)
image_height = loaded_test_image.shape[0]
image_width = loaded_test_image.shape[1]
print(image_height, image_width)

(768, 1024, 3)
768 1024


In [4]:
def get_all_preds_in_test_time_series(block_name, main_data_folder, model):
    # get the path to the block
    path_to_block = os.path.join(main_data_folder, block_name)
    # load the numpy file with the feature sub-widows
    test_features = np.load(os.path.join(path_to_block, [file for file in os.listdir(path_to_block) if file[:9] == 'subwindow'][0]))
    # get the predicted values
    all_predicted_values = model.predict(test_features)

    # return the predicted values
    return(all_predicted_values)

In [5]:
def prediction_on_test_data(pred_values, image_height, image_width, stride = 8, kernel_size = 32):
    # density map
    Density_map = np.zeros((image_height, image_width))

    # counts map
    counts_map = np.zeros((image_height, image_width))
    
    # now, for every window, we will keep adding the values together and also add the counts
    counter = 0
#     need a counter to move into each predicted value in the pred values list
    for ii in range(0, image_height, stride):
        for jj in range(0, image_width, stride):
#         operations for density map
#             get the window of interest
            new_window = Density_map[ii:ii + kernel_size,jj:jj+kernel_size]
#     fill each with the value c_k
            counts_window = np.full((new_window.shape[0], new_window.shape[1]), pred_values[counter])
#     get the shapes of this new window
            cw_height = counts_window.shape[0]
            cw_width = counts_window.shape[1]
#         Do c_k/r_2
            counts_window_new = counts_window/(cw_height*cw_width)
#     This is the value in the window now
            value_window = counts_window_new
#     place the values in the corrsponding area of the density map
            Density_map[ii:ii + kernel_size,jj:jj+kernel_size] = new_window + value_window

#         Let's now focus on capturing the counts of the windows
            new_window_c = counts_map[ii:ii + kernel_size,jj:jj+kernel_size]
#     get the counts area
            count = np.ones((new_window_c.shape[0], new_window_c.shape[1]))
#     keep adding the counts to reflect the addition of densities
            counts_map[ii:ii + kernel_size,jj:jj+kernel_size] = new_window_c + count
#     increase the counter
            counter = counter + 1
            
#         get the normalized count
    normalized_counts = np.divide(Density_map, counts_map)
    
#     entire count on the test set
    pred_on_test = np.sum(normalized_counts)
    
#     return the predicted value
    return(pred_on_test, normalized_counts)


In [6]:
def get_final_forecasted_and_true_values(preds_from_model, im_height, im_weight, stride, kernel_size, csv_file_name, block_name):
    final_preds_list = []
    for i in range(7):
        preds_per_image = prediction_on_test_data(preds_from_model[:,i], im_height, im_weight, stride , kernel_size)
        final_preds_list.append(preds_per_image[0])
    # make this list  a dataframe
    preds_df = pd.DataFrame(final_preds_list, columns = ['Forecasted_value'])
    
    # Where do we have the true values?
    true_val_location = 'All_data/test_true_counts'
    true_value_file = pd.read_csv(os.path.join(true_val_location, csv_file_name))
    
    # compute the mae
    mae_value = mean_absolute_error(true_value_file[['True_count']], preds_df[['Forecasted_value']])
    # compute the rmse
    rmse_value = np.sqrt(mean_squared_error(true_value_file[['True_count']], preds_df[['Forecasted_value']]))
    # pearsonr
    pearson_value = pearsonr(np.array(true_value_file[['True_count']]).reshape(-1), np.array(preds_df[['Forecasted_value']]).reshape(-1))
    # r2score
    r2score_value = r2_score(true_value_file[['True_count']], preds_df[['Forecasted_value']])
    # attach the true and the forecasted values together
    final_df = pd.concat((true_value_file, preds_df), axis = 1)
    # final df location
    final_loc = 'All_data/test_predicted_counts'
    # save this file
    final_df.to_csv(os.path.join(final_loc, block_name + '.csv'), index = False)
    all_metrics = [mae_value, rmse_value, pearson_value, r2score_value]

    return(final_preds_list, all_metrics, final_df)

Block 0103

In [7]:
# first get the predictions
frozen_preds_block_0103 = get_all_preds_in_test_time_series('block_0103', 'All_data', frozen_model)

2025-03-12 10:41:17.182448: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8907


384/384 [==============================] - 5s 6ms/step


In [8]:
frozen_preds_block_0103.shape

(12288, 7)

In [9]:
frozen_final_forecasts_block_0103 = get_final_forecasted_and_true_values(frozen_preds_block_0103, image_height, image_width, 8, 32, 'true_counts_blk_0103.csv', 'from_scratch_block_0103_VGG19')

In [10]:
frozen_normalized_forecasts_block_0103 = frozen_final_forecasts_block_0103[0]

In [11]:
print(frozen_normalized_forecasts_block_0103)

[15.512376707179001, 0.0, 51.15094902706288, 28.705311427828637, 0.0, 14.001849823115208, 0.0]


In [12]:
mae_frozen_block_0103 = frozen_final_forecasts_block_0103[1]
mae_frozen_block_0103

[22.990201581277148,
 25.795500352204968,
 PearsonRResult(statistic=0.4102556571010587, pvalue=0.36062888674128146),
 -23.851359819062296]

In [13]:
frozen_true_forecasted_df = frozen_final_forecasts_block_0103[2]
frozen_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0103_2020_08_26,40,40.000661,15.512377
1,Block0103_2020_08_27,39,39.000001,0.000000
2,Block0103_2020_08_28,41,41.000000,51.150949
3,Block0103_2020_08_31,31,31.000000,28.705311
4,Block0103_2020_09_02,32,32.000000,0.000000
5,Block0103_2020_09_07,40,40.002086,14.001850
6,Block0103_2020_09_16,27,27.000176,0.000000


Block 0104

In [14]:
# first get the predictions
frozen_preds_block_0104 = get_all_preds_in_test_time_series('block_0104', 'All_data', frozen_model)

384/384 [==============================] - 2s 6ms/step


In [15]:
frozen_preds_block_0104.shape

(12288, 7)

In [16]:
frozen_final_forecasts_block_0104 = get_final_forecasted_and_true_values(frozen_preds_block_0104, image_height, image_width, 8, 32, 'true_counts_blk_0104.csv', 'from_scratch_block_0104_VGG19')

In [17]:
frozen_normalized_forecasts_block_0104 = frozen_final_forecasts_block_0104[0]

In [18]:
print(frozen_normalized_forecasts_block_0104)

[8.363541573533439, 0.0, 26.678260386196293, 16.17252727606016, 0.0, 6.229375086340951, 0.0]


In [19]:
mae_frozen_block_0104 = frozen_final_forecasts_block_0104[1]
mae_frozen_block_0104

[28.22232795398131,
 29.488657417564028,
 PearsonRResult(statistic=0.41618790971801795, pvalue=0.3530239935095329),
 -35.73229732606227]

In [20]:
frozen_true_forecasted_df = frozen_final_forecasts_block_0104[2]
frozen_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0104_2020_08_26,33,33.000000,8.363542
1,Block0104_2020_08_27,30,30.000000,0.000000
2,Block0104_2020_08_28,39,39.000001,26.678260
3,Block0104_2020_08_31,40,40.000000,16.172527
4,Block0104_2020_09_02,41,40.998810,0.000000
5,Block0104_2020_09_07,42,42.169009,6.229375
6,Block0104_2020_09_16,30,30.005317,0.000000


Block 0105

In [21]:
# first get the predictions
frozen_preds_block_0105 = get_all_preds_in_test_time_series('block_0105', 'All_data', frozen_model)

384/384 [==============================] - 2s 6ms/step


In [22]:
frozen_preds_block_0105.shape

(12288, 7)

In [23]:
frozen_final_forecasts_block_0105 = get_final_forecasted_and_true_values(frozen_preds_block_0105, image_height, image_width, 8, 32, 'true_counts_blk_0105.csv', 'from_scratch_block_0105_VGG19')

In [24]:
frozen_normalized_forecasts_block_0105 = frozen_final_forecasts_block_0105[0]

In [25]:
print(frozen_normalized_forecasts_block_0105)

[9.318495637094202, 0.0, 43.737071742610475, 20.194794589416535, 0.0, 6.487360681845378, 0.0]


In [26]:
mae_frozen_block_0105 = frozen_final_forecasts_block_0105[1]
mae_frozen_block_0105

[29.323182478433342,
 31.10957198618757,
 PearsonRResult(statistic=0.7165072372458764, pvalue=0.07005707012305393),
 -9.21155641451885]

In [27]:
frozen_true_forecasted_df = frozen_final_forecasts_block_0105[2]
frozen_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0105_2020_08_26,40,40.000001,9.318496
1,Block0105_2020_08_27,46,46.002743,0.000000
2,Block0105_2020_08_28,58,58.000696,43.737072
3,Block0105_2020_08_31,41,41.000032,20.194795
4,Block0105_2020_09_02,41,41.001190,0.000000
5,Block0105_2020_09_07,36,36.000022,6.487361
6,Block0105_2020_09_16,23,23.000000,0.000000


Block 0106

In [28]:
# first get the predictions
frozen_preds_block_0106 = get_all_preds_in_test_time_series('block_0106', 'All_data', frozen_model)

384/384 [==============================] - 2s 6ms/step


In [29]:
frozen_preds_block_0106.shape

(12288, 7)

In [30]:
frozen_final_forecasts_block_0106 = get_final_forecasted_and_true_values(frozen_preds_block_0106, image_height, image_width, 8, 32, 'true_counts_blk_0106.csv', 'from_scratch_block_0106_VGG19')

In [31]:
frozen_normalized_forecasts_block_0106 = frozen_final_forecasts_block_0106[0]

In [32]:
print(frozen_normalized_forecasts_block_0106)

[13.458114629356183, 0.0, 40.775121764962634, 23.88390906228778, 0.0, 12.08112523272776, 0.0]


In [33]:
mae_frozen_block_0106 = frozen_final_forecasts_block_0106[1]
mae_frozen_block_0106

[28.828818472952236,
 31.699777953362922,
 PearsonRResult(statistic=0.38781969142342204, pvalue=0.3899906586766455),
 -83.31321950741983]

In [34]:
frozen_true_forecasted_df = frozen_final_forecasts_block_0106[2]
frozen_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0106_2020_08_26,39,38.999667,13.458115
1,Block0106_2020_08_27,39,38.999989,0.000000
2,Block0106_2020_08_28,45,45.000000,40.775122
3,Block0106_2020_08_31,40,39.986887,23.883909
4,Block0106_2020_09_02,43,42.999997,0.000000
5,Block0106_2020_09_07,48,47.996502,12.081125
6,Block0106_2020_09_16,38,38.000000,0.000000


Block 0201

In [35]:
# first get the predictions
frozen_preds_block_0201 = get_all_preds_in_test_time_series('block_0201', 'All_data', frozen_model)

384/384 [==============================] - 2s 6ms/step


In [36]:
frozen_preds_block_0201.shape

(12288, 7)

In [37]:
frozen_final_forecasts_block_0201 = get_final_forecasted_and_true_values(frozen_preds_block_0201, image_height, image_width, 8, 32, 'true_counts_blk_0201.csv', 'from_scratch_block_0201_VGG19')

In [38]:
frozen_normalized_forecasts_block_0201 = frozen_final_forecasts_block_0201[0]

In [39]:
print(frozen_normalized_forecasts_block_0201)

[14.894463359568666, 0.0, 46.43629275466568, 26.204974303497995, 0.0, 12.849237387881962, 0.0]


In [40]:
mae_frozen_block_0201 = frozen_final_forecasts_block_0201[1]
mae_frozen_block_0201

[25.802147456340812,
 29.68029653718426,
 PearsonRResult(statistic=0.3990993864029589, pvalue=0.3751117613899955),
 -23.387050917640913]

In [41]:
frozen_true_forecasted_df = frozen_final_forecasts_block_0201[2]
frozen_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0201_2020_08_26,45,45.000217,14.894463
1,Block0201_2020_08_27,45,45.000040,0.000000
2,Block0201_2020_08_28,47,47.000001,46.436293
3,Block0201_2020_08_31,38,38.000000,26.204974
4,Block0201_2020_09_02,42,42.000041,0.000000
5,Block0201_2020_09_07,35,35.000000,12.849237
6,Block0201_2020_09_16,29,29.000000,0.000000


Block 0202

In [42]:
# first get the predictions
frozen_preds_block_0202 = get_all_preds_in_test_time_series('block_0202', 'All_data', frozen_model)

384/384 [==============================] - 2s 6ms/step


In [43]:
frozen_preds_block_0202.shape

(12288, 7)

In [44]:
frozen_final_forecasts_block_0202 = get_final_forecasted_and_true_values(frozen_preds_block_0202, image_height, image_width, 8, 32, 'true_counts_blk_0202.csv', 'from_scratch_block_0202_VGG19')

In [45]:
frozen_normalized_forecasts_block_0202 = frozen_final_forecasts_block_0202[0]

In [46]:
print(frozen_normalized_forecasts_block_0202)

[18.356930057329162, 0.0, 60.46889283436636, 35.23924940180753, 0.0, 15.334691424534261, 0.0]


In [47]:
mae_frozen_block_0202 = frozen_final_forecasts_block_0202[1]
mae_frozen_block_0202

[16.390054409852684,
 20.070964894694395,
 PearsonRResult(statistic=0.5491419348156342, pvalue=0.2017017989233782),
 -116.49605927618262]

In [48]:
frozen_true_forecasted_df = frozen_final_forecasts_block_0202[2]
frozen_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0202_2020_08_26,18,17.999960,18.356930
1,Block0202_2020_08_27,21,21.000000,0.000000
2,Block0202_2020_08_28,23,23.000000,60.468893
3,Block0202_2020_08_31,21,20.999982,35.239249
4,Block0202_2020_09_02,21,21.000000,0.000000
5,Block0202_2020_09_07,18,18.000000,15.334691
6,Block0202_2020_09_16,18,18.000000,0.000000


Block 0205

In [49]:
# first get the predictions
frozen_preds_block_0205 = get_all_preds_in_test_time_series('block_0205', 'All_data', frozen_model)

384/384 [==============================] - 2s 6ms/step


In [50]:
frozen_preds_block_0205.shape

(12288, 7)

In [51]:
frozen_final_forecasts_block_0205 = get_final_forecasted_and_true_values(frozen_preds_block_0205, image_height, image_width, 8, 32, 'true_counts_blk_0205.csv', 'from_scratch_block_0205_VGG19')

In [52]:
frozen_normalized_forecasts_block_0205 = frozen_final_forecasts_block_0205[0]

In [53]:
print(frozen_normalized_forecasts_block_0205)

[15.046954869651625, 0.0, 42.292923713899654, 24.567305051630385, 7.19810341252014e-05, 13.898423035114925, 0.0]


In [54]:
mae_frozen_block_0205 = frozen_final_forecasts_block_0205[1]
mae_frozen_block_0205

[27.170617335524184,
 29.83014100552497,
 PearsonRResult(statistic=0.667546296222285, pvalue=0.10132210008606322),
 -53.231378492618916]

In [55]:
frozen_true_forecasted_df = frozen_final_forecasts_block_0205[2]
frozen_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0205_2020_08_26,44,44.000000,15.046955
1,Block0205_2020_08_27,42,42.000001,0.000000
2,Block0205_2020_08_28,45,45.000000,42.292924
3,Block0205_2020_08_31,43,43.000000,24.567305
4,Block0205_2020_09_02,39,39.000000,0.000072
5,Block0205_2020_09_07,41,40.999915,13.898423
6,Block0205_2020_09_16,32,31.999656,0.000000


Block 0206

In [56]:
# first get the predictions
frozen_preds_block_0206 = get_all_preds_in_test_time_series('block_0206', 'All_data', frozen_model)

384/384 [==============================] - 2s 6ms/step


In [57]:
frozen_preds_block_0206.shape

(12288, 7)

In [58]:
frozen_final_forecasts_block_0206 = get_final_forecasted_and_true_values(frozen_preds_block_0206, image_height, image_width, 8, 32, 'true_counts_blk_0206.csv', 'from_scratch_block_0206_VGG19')

In [59]:
frozen_normalized_forecasts_block_0206 = frozen_final_forecasts_block_0206[0]

In [60]:
print(frozen_normalized_forecasts_block_0206)

[15.75056251696939, 0.0, 54.48039442012305, 28.249001291881846, 0.0, 14.136029362371726, 0.0]


In [61]:
mae_frozen_block_0206 = frozen_final_forecasts_block_0206[1]
mae_frozen_block_0206

[19.763543035557152,
 22.937233210235423,
 PearsonRResult(statistic=0.39905855329465645, pvalue=0.3751651994573756),
 -5.651113699611873]

In [62]:
frozen_true_forecasted_df = frozen_final_forecasts_block_0206[2]
frozen_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0206_2020_08_26,41,40.999838,15.750563
1,Block0206_2020_08_27,42,41.998810,0.000000
2,Block0206_2020_08_28,39,39.000067,54.480394
3,Block0206_2020_08_31,32,32.000003,28.249001
4,Block0206_2020_09_02,25,25.000352,0.000000
5,Block0206_2020_09_07,23,23.000040,14.136029
6,Block0206_2020_09_16,18,18.000000,0.000000


Block 0302

In [63]:
# first get the predictions
frozen_preds_block_0302 = get_all_preds_in_test_time_series('block_0302', 'All_data', frozen_model)

384/384 [==============================] - 2s 6ms/step


In [64]:
frozen_preds_block_0302.shape

(12288, 7)

In [65]:
frozen_final_forecasts_block_0302 = get_final_forecasted_and_true_values(frozen_preds_block_0302, image_height, image_width, 8, 32, 'true_counts_blk_0302.csv', 'from_scratch_block_0302_VGG19')

In [66]:
frozen_normalized_forecasts_block_0302 = frozen_final_forecasts_block_0302[0]

In [67]:
print(frozen_normalized_forecasts_block_0302)

[16.15776326440876, 0.0, 49.664100260603945, 28.085090563422654, 0.0, 14.13452566501898, 0.0]


In [68]:
mae_frozen_block_0302 = frozen_final_forecasts_block_0302[1]
mae_frozen_block_0302

[31.70836003522081,
 34.52441402811589,
 PearsonRResult(statistic=0.7563490920624635, pvalue=0.04911319312306086),
 -44.628767996291806]

In [69]:
frozen_true_forecasted_df = frozen_final_forecasts_block_0302[2]
frozen_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0302_2020_08_26,49,49.000000,16.157763
1,Block0302_2020_08_27,49,49.000005,0.000000
2,Block0302_2020_08_28,54,53.999570,49.664100
3,Block0302_2020_08_31,50,50.042349,28.085091
4,Block0302_2020_09_02,43,43.000009,0.000000
5,Block0302_2020_09_07,48,48.000572,14.134526
6,Block0302_2020_09_16,37,36.999999,0.000000


Block 0303

In [70]:
# first get the predictions
frozen_preds_block_0303 = get_all_preds_in_test_time_series('block_0303', 'All_data', frozen_model)

384/384 [==============================] - 2s 6ms/step


In [71]:
frozen_preds_block_0303.shape

(12288, 7)

In [72]:
frozen_final_forecasts_block_0303 = get_final_forecasted_and_true_values(frozen_preds_block_0303, image_height, image_width, 8, 32, 'true_counts_blk_0303.csv', 'from_scratch_block_0303_VGG19')

In [73]:
frozen_normalized_forecasts_block_0303 = frozen_final_forecasts_block_0303[0]

In [74]:
print(frozen_normalized_forecasts_block_0303)

[12.746222730339467, 0.0, 41.76642949637849, 23.125902107809658, 0.0, 9.608461588144564, 0.0]


In [75]:
mae_frozen_block_0303 = frozen_final_forecasts_block_0303[1]
mae_frozen_block_0303

[26.393283439618262,
 29.773810030535326,
 PearsonRResult(statistic=0.3491157646563067, pvalue=0.4427707009606065),
 -13.139813939774053]

In [76]:
frozen_true_forecasted_df = frozen_final_forecasts_block_0303[2]
frozen_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0303_2020_08_26,49,49.000171,12.746223
1,Block0303_2020_08_27,46,46.000007,0.000000
2,Block0303_2020_08_28,43,42.999982,41.766429
3,Block0303_2020_08_31,39,38.999993,23.125902
4,Block0303_2020_09_02,36,36.025884,0.000000
5,Block0303_2020_09_07,36,36.000000,9.608462
6,Block0303_2020_09_16,23,23.000000,0.000000


Block 0304

In [77]:
# first get the predictions
frozen_preds_block_0304 = get_all_preds_in_test_time_series('block_0304', 'All_data', frozen_model)

384/384 [==============================] - 2s 6ms/step


In [78]:
frozen_preds_block_0304.shape

(12288, 7)

In [79]:
frozen_final_forecasts_block_0304 = get_final_forecasted_and_true_values(frozen_preds_block_0304, image_height, image_width, 8, 32, 'true_counts_blk_0304.csv', 'from_scratch_block_0304_VGG19')

In [80]:
frozen_normalized_forecasts_block_0304 = frozen_final_forecasts_block_0304[0]

In [81]:
print(frozen_normalized_forecasts_block_0304)

[14.589045409320534, 0.0, 42.695939150630785, 25.09387562037027, 0.0, 10.927647158741214, 0.0]


In [82]:
mae_frozen_block_0304 = frozen_final_forecasts_block_0304[1]
mae_frozen_block_0304

[23.670498951562454,
 26.783507307500876,
 PearsonRResult(statistic=0.5554032935539531, pvalue=0.195543110880948),
 -18.61521033529949]

In [83]:
frozen_true_forecasted_df = frozen_final_forecasts_block_0304[2]
frozen_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0304_2020_08_26,37,37.000000,14.589045
1,Block0304_2020_08_27,41,41.002057,0.000000
2,Block0304_2020_08_28,43,42.999998,42.695939
3,Block0304_2020_08_31,42,41.999955,25.093876
4,Block0304_2020_09_02,38,38.000000,0.000000
5,Block0304_2020_09_07,34,34.000000,10.927647
6,Block0304_2020_09_16,24,24.000000,0.000000


Block 0305

In [84]:
# first get the predictions
frozen_preds_block_0305 = get_all_preds_in_test_time_series('block_0305', 'All_data', frozen_model)

384/384 [==============================] - 2s 6ms/step


In [85]:
frozen_preds_block_0305.shape

(12288, 7)

In [86]:
frozen_final_forecasts_block_0305 = get_final_forecasted_and_true_values(frozen_preds_block_0305, image_height, image_width, 8, 32, 'true_counts_blk_0305.csv', 'from_scratch_block_0305_VGG19')

In [87]:
frozen_normalized_forecasts_block_0305 = frozen_final_forecasts_block_0305[0]

In [88]:
print(frozen_normalized_forecasts_block_0305)

[21.61196168618436, 0.0, 67.27840637429433, 36.159022350159404, 0.0, 18.706372336807867, 0.0]


In [89]:
mae_frozen_block_0305 = frozen_final_forecasts_block_0305[1]
mae_frozen_block_0305

[23.017013528780215,
 25.369978411335047,
 PearsonRResult(statistic=0.28944740416491277, pvalue=0.5289414107674056),
 -10.646290408046053]

In [90]:
frozen_true_forecasted_df = frozen_final_forecasts_block_0305[2]
frozen_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0305_2020_08_26,46,46.000000,21.611962
1,Block0305_2020_08_27,35,35.000000,0.000000
2,Block0305_2020_08_28,37,37.000000,67.278406
3,Block0305_2020_08_31,30,30.000652,36.159022
4,Block0305_2020_09_02,35,35.001190,0.000000
5,Block0305_2020_09_07,29,28.999994,18.706372
6,Block0305_2020_09_16,20,20.000018,0.000000


Block 0306

In [91]:
# first get the predictions
frozen_preds_block_0306 = get_all_preds_in_test_time_series('block_0306', 'All_data', frozen_model)

384/384 [==============================] - 2s 6ms/step


In [92]:
frozen_preds_block_0306.shape

(12288, 7)

In [93]:
frozen_final_forecasts_block_0306 = get_final_forecasted_and_true_values(frozen_preds_block_0306, image_height, image_width, 8, 32, 'true_counts_blk_0306.csv', 'from_scratch_block_0306_VGG19')

In [94]:
frozen_normalized_forecasts_block_0306 = frozen_final_forecasts_block_0306[0]

In [95]:
print(frozen_normalized_forecasts_block_0306)

[16.95959757990886, 0.0, 51.35309936504291, 29.409907540037842, 0.0, 14.72363783721812, 0.0]


In [96]:
mae_frozen_block_0306 = frozen_final_forecasts_block_0306[1]
mae_frozen_block_0306

[22.89427948683973,
 25.90888068855905,
 PearsonRResult(statistic=0.44992981595259596, pvalue=0.31108065860350087),
 -9.177052855249201]

In [97]:
frozen_true_forecasted_df = frozen_final_forecasts_block_0306[2]
frozen_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0306_2020_08_26,41,41.000009,16.959598
1,Block0306_2020_08_27,41,41.000003,0.000000
2,Block0306_2020_08_28,43,43.006851,51.353099
3,Block0306_2020_08_31,40,39.997604,29.409908
4,Block0306_2020_09_02,40,40.000000,0.000000
5,Block0306_2020_09_07,33,32.999982,14.723638
6,Block0306_2020_09_16,18,18.000000,0.000000
